# Kombinierter LSTM: Bidirektionales Encoding

## Fragestellung

Ein Spektrum ist entlang der Wellenlänge geordnet. Ein bidirektionaler LSTM verarbeitet diese Folge einmal von kurzen zu langen und einmal von langen zu kurzen Wellenlängen. Dieses Notebook prüft auf den fünf Trainingsfolds, ob beide Richtungen gemeinsam die kombinierte Balanced Accuracy verbessern.

Der gemeinsame Holdout wurde für die LSTM-Familie bereits ausgewertet. Deshalb enthält dieses Notebook bewusst keinen `run()`-Aufruf und erzeugt keine neue Holdout-Metrik.

## Versuchsdesign

Das Raster vergleicht den bestehenden unidirektionalen Encoder direkt mit einem bidirektionalen Encoder. Hidden Size, LSTM-Schichten, Early Stopping, AEZ-/Month-Embeddings, Vorverarbeitung und CV-Folds bleiben gleich. Der bidirektionale Kandidat kombiniert die letzten Zustände beider Wellenlängenrichtungen erst vor der Fusion mit dem Kontext.

In [ ]:
import importlib.metadata
import platform
import subprocess

import numpy as np
import torch
from IPython.display import display

from awp2.config import LSTM_EMBEDDINGS_CONTEXT, SEED
from awp2.experiment import TuneConfig, tune
from awp2.models import make_combined_spectral_lstm
from awp2.preprocessing import PreprocessingConfig

EXPERIMENT_ID = "combined_lstm_bidirectional"
TUNING_ID = f"{EXPERIMENT_ID}_search"
PREPROCESSING = PreprocessingConfig(
    use_spectral_standard_scale=True,
    use_categorical_context=True,
)
PARAM_GRID = {
    "estimator__bidirectional": [False, True],
}

np.random.seed(SEED)
torch.manual_seed(SEED)
environment = {
    "python": platform.python_version(),
    "torch": importlib.metadata.version("torch"),
    "scikit_learn": importlib.metadata.version("scikit-learn"),
    "branch": subprocess.check_output(
        ["git", "branch", "--show-current"],
        text=True,
    ).strip(),
    "preprocessing": PREPROCESSING.model_dump(),
    "param_grid": PARAM_GRID,
}
display(environment)

tuned = tune(
    make_combined_spectral_lstm(LSTM_EMBEDDINGS_CONTEXT),
    TuneConfig(
        name=TUNING_ID,
        approach="combined",
        description=(
            "CV-only: Vergleich eines uni- und bidirektionalen kombinierten LSTM "
            "mit AEZ- und Monatsembeddings."
        ),
        preprocessing=PREPROCESSING,
        param_grid=PARAM_GRID,
    ),
)
results = tuned.candidates.sort_values("rank")
display(results)
{
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "tuning_run": tuned.run_id,
}

## Auswertung

Vergleiche die mittlere CV-Score und die Fold-Streuung beider Richtungen. Ein Vorsprung unter `MIN_CV_GAIN = 0.005` gilt als Rauschen. Berichte die Kandidaten und die Entscheidung als CV-only-Ergebnis in MLflow und im Experiment-Log.

Auch bei einem klaren CV-Gewinn wird kein weiterer Holdout-Lauf gestartet: Der gemeinsame Holdout wurde für die LSTM-Familie bereits verwendet.